In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.ibdb.com/broadway-production/oklahoma-1285"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

In [ ]:
with open("../data/raw/oklahoma_page.html", "w", encoding="utf-8") as f:
    f.write(response.text)

In [ ]:
from bs4 import BeautifulSoup

with open("../data/raw/oklahoma_page.html", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

In [ ]:
print(soup.title)

h1 = soup.find("h1")
print(h1)

In [ ]:
for tag in soup.find_all(["h1", "h2", "h3"]):
    print(tag.name, ":", tag.get_text(strip=True))

In [ ]:
matches = soup.find_all(string=lambda s: s and "Alfred Drake" in s)

len(matches)

In [ ]:
matches[0]


In [ ]:

matches[0].parent


In [ ]:

matches[0].parent.prettify()

In [ ]:
title = soup.find(string="Oklahoma!")

print(title.parent.prettify())

In [ ]:
alfred = matches[0]

for i in range(1, 6):
    print(f"\nLEVEL {i}")
    print(alfred.parent)
    alfred = alfred.parent.parent

In [ ]:
matches[0].find_parent().parent.parent.prettify()

In [ ]:
cast_heading = soup.find(string=lambda x: x and "Opening Night Cast" in x)

print(cast_heading)

In [ ]:
print(cast_heading.parent.prettify())

In [ ]:
print(cast_heading.parent.parent.prettify())

In [ ]:
print(cast_heading.parent.parent.prettify()[:3000])

In [ ]:
rows = soup.find_all("div", class_="row mobile-a-align")

print(len(rows))

In [ ]:
for row in rows[:5]:
    print(row.get_text(" | ", strip=True))

In [ ]:
opening_cast = soup.find(id="OpeningNightCast")

print(opening_cast)

In [ ]:
print(opening_cast.prettify()[:3000])

In [ ]:
rows = opening_cast.find_all("div", class_="row mobile-a-align")

print(len(rows))

for row in rows[:10]:
    cols = row.find_all("div", class_="col m4 s12")
    print(
        cols[0].get_text(strip=True),
        "|",
        cols[1].get_text(" ", strip=True)
    )

In [ ]:
for row in rows[:10]:
    cols = row.find_all("div", class_="col m4 s12")

    person = cols[0].find("a")

    if person:
        print(
            person.get_text(strip=True),
            "|",
            person["href"]
        )

In [32]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin


def extract_opening_cast(soup):
    """
    Extract opening night cast from an IBDB production page.

    Returns:
        list of dictionaries containing performer information
    """

    cast_section = soup.find(id="OpeningNightCast")

    if cast_section is None:
        return []

    cast = []

    rows = cast_section.find_all("div", class_="row mobile-a-align")

    for row in rows:
        cols = row.find_all("div", recursive=False)

        if len(cols) < 2:
            continue

        performer_link = cols[0].find("a")

        # Skip ensemble/non-person entries
        if performer_link is None:
            continue

        performer_name = performer_link.get_text(strip=True)
        performer_url = performer_link["href"]

        performer_id = performer_url.rstrip("/").split("-")[-1]

        character = cols[1].get_text(strip=True)

        cast.append({
            "performer_id": performer_id,
            "performer_name": performer_name,
            "performer_url": performer_url,
            "character": character
        })

    return cast

In [33]:
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

cast = extract_opening_cast(soup)

cast[:5]

[{'performer_id': '4031',
  'performer_name': 'Alfred Drake',
  'performer_url': '/broadway-cast-staff/alfred-drake-4031',
  'character': 'Curly'},
 {'performer_id': '57977',
  'performer_name': 'Joan Roberts',
  'performer_url': '/broadway-cast-staff/joan-roberts-57977',
  'character': 'Laurey'},
 {'performer_id': '14313',
  'performer_name': 'Joseph Buloff',
  'performer_url': '/broadway-cast-staff/joseph-buloff-14313',
  'character': 'Ali Hakim'},
 {'performer_id': '67218',
  'performer_name': 'Howard Da Silva',
  'performer_url': '/broadway-cast-staff/howard-da-silva-67218',
  'character': 'Jud Fry'},
 {'performer_id': '38152',
  'performer_name': 'Lee Dixon',
  'performer_url': '/broadway-cast-staff/lee-dixon-38152',
  'character': 'Will Parker'}]

In [35]:
for person in cast:
    print(person["performer_name"], "-", person["character"])

Alfred Drake - Curly
Joan Roberts - Laurey
Joseph Buloff - Ali Hakim
Howard Da Silva - Jud Fry
Lee Dixon - Will Parker
Betty Garde - Aunt Eller
Celeste Holm - Ado Annie Carnes
Remo Arlotta - Singing Ensemble
Elsie Arnold - Singing Ensemble
John Baum - Singing Ensemble
Harvey Brown - Singing Ensemble
Kenneth Buffett - Dancing Ensemble
Joseph Cunneff - Singing Ensemble
Jack Dunphy - Dancing Ensemble
Gary Fleming - Dancing Ensemble
June Graham - Laurey's FriendDream Ballet
Ray Harrison - Dancing Ensemble
Jack Harwood - A Cowboy
Edmund Howland - Dancing Ensemble
Barry Kelley - Ike Skidmore
Eric Kristen - Dancing Ensemble
Jane Lawrence - Gertie Cummings
Suzanne Lloyd - Singing Ensemble
Dorothea MacFarland - Singing Ensemble
Owen Martin - Cord Elam
May Muth - Singing Ensemble
Carl Nelson - Singing Ensemble
Virginia Oswald - Singing Ensemble
Robert Penn - Singing Ensemble
Ralph Riggs - Andrew Carnes
Rosemary Schaefer - Laurey's FriendDream Ballet
Vivienne Simon - Singing Ensemble
Faye Smith -

# Working extraction functions
- extract_opening_cast() has now been validated

# Production Metadata Extraction

## Fields to investigate

□ production_id
□ title
□ opening_date
□ year
□ production type (Musical / Play)
□ original vs revival

In [36]:
# Look for common production metadata labels

keywords = [
    "Opening",
    "Opened",
    "Preview",
    "Closing",
    "Play",
    "Musical",
    "Revival"
]

for keyword in keywords:
    print(f"\n=== Searching for '{keyword}' ===")

    matches = soup.find_all(string=lambda text: text and keyword.lower() in text.lower())

    if not matches:
        print("No matches.")
        continue

    for match in matches[:10]:
        print(match.strip())


=== Searching for 'Opening' ===
Opening Date
Opening Night Cast
Replaced an injured dancer a day after the show's opening
Opening Night Cast
Replaced an injured dancer a day after the show's opening
More Productions by Opening Date

=== Searching for 'Opened' ===
No matches.

=== Searching for 'Preview' ===
No matches.

=== Searching for 'Closing' ===
To collect end-user usage analytics about your application,
    insert the following script into each page you want to track.
    Place this code immediately before the closing </head> tag,
    and before any other scripts. Your first data will appear
    automatically in just a few seconds.
Closing Date
Closing date unknown

=== Searching for 'Play' ===
function isImageBroken(img) {
      return img.complete && img.naturalWidth === 0;
    }

    window.addEventListener('load', () => {
        const imgs = $('.img-thumb-gallery').find('img.responsive-img').toArray();
        const fullImgs = $('.img-large-gallery').find('img.responsive-i

In [37]:
# Find the "Opening Date" label and inspect its surrounding HTML

opening_label = soup.find(string=lambda text: text and text.strip() == "Opening Date")

if opening_label:
    print(opening_label.parent.prettify())
else:
    print("Opening Date label not found.")

<div class="xt-lable">
 Opening Date
</div>



In [38]:
opening_label = soup.find(string=lambda text: text and text.strip() == "Opening Date")

if opening_label:
    print(opening_label.parent.parent.prettify())

<div class="col s5 m3 l5 txt-paddings">
 <div class="xt-lable">
  Opening Date
 </div>
 <div class="xt-main-title">
  Mar 31, 1943
 </div>
</div>



In [39]:
musical = soup.find(string=lambda text: text and text.strip() == "Musical")

if musical:
    print(musical.parent.prettify())

<i>
 Musical
</i>



In [40]:
musical = soup.find(string=lambda text: text and text.strip() == "Musical")

if musical:
    print(musical.parent.parent.prettify())

<div class="col s12 txt-paddings tag-block-compact">
 <i>
  Musical
 </i>
 <i>
  Comedy
 </i>
 <i>
  Original
 </i>
 <i>
  Broadway
 </i>
</div>



In [41]:
# Print every metadata label/value pair on the page

for label in soup.find_all("div", class_="xt-lable"):
    
    value = label.find_next_sibling("div")
    
    print(f"{label.get_text(strip=True):20} -> {value.get_text(' ', strip=True) if value else 'None'}")

Opening Date         -> Mar 31, 1943
Closing Date         -> May 29, 1948
Performances         -> 2,212


In [44]:
title = soup.find("h3", class_="title-label")

print(title.parent.parent.prettify())

<div class="title">
 <div>
  <h3 class="title-label">
   Oklahoma!
  </h3>
 </div>
</div>



In [43]:
print(soup.title.get_text(strip=True))

Oklahoma! – Broadway Musical – Original | IBDB


In [48]:
url = "https://www.ibdb.com/broadway-production/she-loves-me-501744"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.get_text(strip=True))

She Loves Me – Broadway Musical – 2016 Revival | IBDB
